# 레이더 인체 탐지 모델 학습 — **Cascade v2** (pobok 구제)

기존 cascade 모델 (v1, output_cas/) 의 핵심 실패 모드 해결:
- **증상**: 포복/엎드림 사람은 박스는 잡히지만 `human_prob < 0.5` 로 강아지로 오분류.
- **원인**: metadata 11 차원의 `z_range` / `v_std` / `z_max` 가 shortcut 학습을 유발.
  PointNet/GRU 가 spatial/temporal 패턴을 학습하지 않고 11 차원 통계만으로 결정.

## v2 주요 변경 (handoff 문서 기준)

| 항목 | v1 | v2 | 효과 |
|---|---|---|---|
| METADATA_DIM | 11 | **8** | shortcut 차단 — z_max, z_range, v_std 제거 |
| Z-scale aug | X | **±50%** | 사람의 z 폭 다양성 강제 |
| Pobok sampler | X | **(h=1)∩(p=2) × 4.0** | 포복 학습 비중 ↑ |
| 평가 | 전체 | **+ 세션별 + class별 + pobok/dog 합격** | 회귀 감지 |

## 합격 조건 (handoff §7)

- val ROC-AUC ≥ v1 수준 (회귀 없음)
- **pobok recall (sigmoid > 0.5) ≥ 0.70** ← 핵심
- **dog FPR < 0.10** (강아지를 사람으로 분류 안 함)
- ARM 라이브 검증 통과 (별도)

## 산출물 (handoff §8)

- `model.tflite` (Int8) — `output_cas_v2/embed/model_int8.tflite`
- `tfmodel.h` — C array header
- `embed_report.json` — parity 검증
- `eval_report.json` — 세션별 + class별 + pobok·dog 합격 결과

## ⚠ C++ 동기화 필수

METADATA_DIM 8 로 모델 추출 후, 다음 파일 동기 수정 필요:
- `radar_embed/include/radar/config.h:71` → `METADATA_DIM = 8`
- `radar_embed/src/metadata.cc::extract()` → 8 차원만 채움 (순서 명시: 셀 6)


## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 환경 확인 + 의존성

In [ ]:
import torch, sys, numpy as np
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
print('numpy  :', np.__version__)

try:
    from torch.utils.tensorboard import SummaryWriter
    print('tensorboard: OK')
except ImportError:
    !pip install -q tensorboard


## 3. 설정

v2 출력은 `output_cas_v2/` — v1 (`output_cas/`) 과 분리 (성능 비교용).


In [ ]:
from pathlib import Path

DATA_MODE  = 'npz_local'

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_DIR   = DRIVE_ROOT / 'dataset'
NPZ_NAME   = 'all_sequences_v7.npz'
ZIP_NAME   = 'preprocessed_data.zip'
LOCAL_ROOT = Path('/content/radar_data')

# v2 전용 출력 — v1 결과와 분리해서 비교 가능하게.
OUT_DIR = DRIVE_ROOT / 'output_cas_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============ 하이퍼파라미터 ============
EPOCHS         = 150
BATCH_SIZE     = 32
LR             = 1e-3
WEIGHT_DECAY   = 3e-4
WEIGHT_HUMAN   = 0.6
WEIGHT_POSE    = 0.4
PATIENCE       = 40
MIN_EPOCHS     = 50
GRAD_CLIP      = 1.0
THRESHOLD      = 0.5         # v1 의 0.7 → 0.5 (handoff: alarm threshold = 0.5)
POSE_LABEL_SMOOTH = 0.05
SEED           = 42
EARLY_STOP_METRIC = 'combined'

# Augmentation
AUG_JITTER_STD     = 0.03
AUG_DROPOUT_P      = 0.3
AUG_DOPPLER_SCALE  = 0.08
AUG_YAW_RANDOM     = True
ROI_RADIUS         = 1.2

# === v2 신규: Z-scale aug ===
# 학습 sample 마다 z 좌표 ×U(AUG_Z_SCALE_MIN, AUG_Z_SCALE_MAX).
# 0.5 = 사람을 절반 키로, 1.5 = 1.5배 키로 보임.
# → 모델이 z 절대 크기에 의존 못 하고 spatial/temporal 패턴 학습 강제.
AUG_Z_SCALE_MIN = 0.5
AUG_Z_SCALE_MAX = 1.5

# === Sampler — v2 신규 가중치 ===
W_CLASS_HUMAN    = 1.0       # 보통 사람 (서있음/누움)
W_CLASS_ANIMAL   = 12.0      # 강아지
W_CLASS_CLUTTER  = 8.0       # 노이즈
W_CLASS_LOW_HUMAN = 4.0      # 신규: (human=1) & (pose==2) — pobok/엎드림 사람
                             # 4× 가중치 — 1,849 sample 이 train 의 ~10% 인데
                             # ×4 = ~30~40% effective 비중. 다른 클래스 분포는 거의 유지.

# Focal Loss
USE_FOCAL_LOSS  = True
FOCAL_ALPHA     = 0.2
FOCAL_GAMMA     = 2.0

# === Cascade — v2 핵심 변경: METADATA_DIM 11 → 8 ===
METADATA_DIM    = 8           # was 11. 제거: z_max, z_range, v_std
HEAD_HIDDEN     = 64

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'\n[v2] METADATA_DIM = {METADATA_DIM}  (v1: 11)')
print(f'[v2] z-scale aug = [{AUG_Z_SCALE_MIN}, {AUG_Z_SCALE_MAX}]')
print(f'[v2] W_LOW_HUMAN = {W_CLASS_LOW_HUMAN}  (pobok 가중치)')


## 4. 데이터 준비 — DATA_MODE 분기

In [ ]:
import time, shutil, zipfile

t0 = time.time()
if DATA_MODE == 'npz_drive':
    DATA_PATH = DATA_DIR / NPZ_NAME
    print(f'using NPZ directly from Drive: {DATA_PATH}')
elif DATA_MODE == 'npz_local':
    src = DATA_DIR / NPZ_NAME
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    DATA_PATH = LOCAL_ROOT / NPZ_NAME
    if not DATA_PATH.exists() or DATA_PATH.stat().st_size != src.stat().st_size:
        print(f'copying {src.name} → {DATA_PATH} ({src.stat().st_size/1024/1024:.1f} MB)...')
        shutil.copy2(src, DATA_PATH)
    else:
        print(f'cached: {DATA_PATH}')
elif DATA_MODE == 'zip_local':
    src_zip = DATA_DIR / ZIP_NAME
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    cands = list(LOCAL_ROOT.rglob(NPZ_NAME))
    if not cands:
        print(f'extracting {src_zip.name}...')
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(LOCAL_ROOT)
        cands = list(LOCAL_ROOT.rglob(NPZ_NAME))
    if not cands:
        raise FileNotFoundError(f'{NPZ_NAME} not found inside {src_zip}')
    DATA_PATH = cands[0]
    print(f'NPZ located: {DATA_PATH}')
else:
    raise ValueError(f'Unknown DATA_MODE: {DATA_MODE}')

assert DATA_PATH.exists(), f'NPZ 없음: {DATA_PATH}'
print(f'NPZ size : {DATA_PATH.stat().st_size/1024/1024:.1f} MB')
print(f'prepared in {time.time()-t0:.1f}s')


## 5. 모델 정의 — Cascade (METADATA_DIM=8)

v1 과 구조 동일, 다만 metadata 차원만 11 → 8 로 축소. human_head 입력 = 128+3+8 = 139차원.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class PointNetEncoder(nn.Module):
    def __init__(self, in_dim=5, out_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv1d(in_dim, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, 1, bias=False),    nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Conv1d(128, out_dim, 1, bias=False), nn.BatchNorm1d(out_dim), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.mlp(x)
        x, _ = x.max(dim=2)
        return x


class HumanPoseModelCascade(nn.Module):
    def __init__(self, in_dim=5, pointnet_dim=256, gru_hidden=128,
                 n_poses=3, head_hidden=64, dropout=0.3, metadata_dim=8):
        super().__init__()
        self.encoder = PointNetEncoder(in_dim, pointnet_dim)
        self.gru = nn.GRU(input_size=pointnet_dim, hidden_size=gru_hidden,
                          num_layers=1, batch_first=True)
        self.pose_head = nn.Sequential(
            nn.Linear(gru_hidden, head_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(head_hidden, n_poses),
        )
        human_in_dim = gru_hidden + n_poses + metadata_dim
        self.human_head = nn.Sequential(
            nn.Linear(human_in_dim, head_hidden * 2), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(head_hidden * 2, head_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x, meta):
        B, T, N, Fdim = x.shape
        x = x.reshape(B * T, N, Fdim)
        feat = self.encoder(x)
        feat = feat.reshape(B, T, -1)
        gru_out, _ = self.gru(feat)
        backbone = gru_out[:, -1, :]
        pose_logits = self.pose_head(backbone)
        pose_probs  = F.softmax(pose_logits, dim=-1)
        human_input = torch.cat([backbone, pose_probs, meta], dim=-1)
        human_logit = self.human_head(human_input).squeeze(-1)
        return human_logit, pose_logits


_m = HumanPoseModelCascade(metadata_dim=METADATA_DIM).to(DEVICE)
_n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'model params: {_n_params:,}')
_x = torch.randn(2, 40, 64, 5, device=DEVICE)
_meta = torch.randn(2, METADATA_DIM, device=DEVICE)
_h, _p = _m(_x, _meta)
print(f'  input  : pts {tuple(_x.shape)}, meta {tuple(_meta.shape)}')
print(f'  human  : {tuple(_h.shape)}')
print(f'  pose   : {tuple(_p.shape)}')
del _m, _x, _meta, _h, _p


## 6. Dataset — Metadata 8차원 + Z-scale aug

### v2 metadata 8 차원 (C++ metadata.cc 와 1:1 일치 필수)

| idx | 의미 | v1 idx |
|---|---|---|
| 0 | z 평균 | 0 |
| 1 | z 표준편차 | 1 |
| 2 | z min (지면 proxy) | 3 |
| 3 | \|V\| 평균 (속도 proxy) | 5 |
| 4 | P 평균 (반사강도 proxy) | 7 |
| 5 | P 표준편차 | 8 |
| 6 | x spread (frame별 x.std 평균) | 9 |
| 7 | y spread (frame별 y.std 평균) | 10 |

**제거됨**: z_max (v1 idx 2), z_range (v1 idx 4), v_std (v1 idx 6)
→ "키 크고 다리 흔드는 것 = 사람" shortcut 차단.

### v2 Augmentation 추가

- `aug_z_scale_range`: 매 sample 마다 z 좌표 ×U(0.5, 1.5)
- metadata 는 augment **후** points 로 재계산 → 학습/추론 정합 유지.


In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def extract_metadata_np(pts: np.ndarray) -> np.ndarray:
    """v2: pts (T, N, 5) → metadata (8,) float32.

    순서 (radar_embed/src/metadata.cc 와 1:1 일치 유지):
      [z_mean, z_std, z_min, |v|_mean, p_mean, p_std, x_spread, y_spread]
    """
    z = pts[..., 2]
    v = pts[..., 3]
    p = pts[..., 4]
    return np.array([
        z.mean(),
        z.std(),
        z.min(),
        np.abs(v).mean(),
        p.mean(),
        p.std(),
        pts[..., 0].std(axis=1).mean(),
        pts[..., 1].std(axis=1).mean(),
    ], dtype=np.float32)


class SequenceDatasetCascade(Dataset):
    def __init__(self, npz_path, indices=None, augment=False,
                 aug_jitter_std=0.02, aug_dropout_p=0.2,
                 aug_doppler_scale=0.05, aug_yaw_random=True, roi_radius=1.2,
                 aug_z_scale_range=None):
        data = np.load(npz_path, allow_pickle=True)
        self.points = np.asarray(data['points'], dtype=np.float32)
        self.human  = np.asarray(data['human'],  dtype=np.int64)
        self.pose   = np.asarray(data['pose'],   dtype=np.int64)
        if indices is not None:
            indices = np.asarray(indices)
            self.points = self.points[indices]
            self.human  = self.human[indices]
            self.pose   = self.pose[indices]
        self.augment = augment
        self.aug_jitter_std = aug_jitter_std
        self.aug_dropout_p = aug_dropout_p
        self.aug_doppler_scale = aug_doppler_scale
        self.aug_yaw_random = aug_yaw_random
        self.roi_radius = roi_radius
        self.aug_z_scale_range = aug_z_scale_range   # v2 신규

    def __len__(self):
        return len(self.human)

    def _augment(self, pts):
        out = pts.copy()

        # 1. Yaw (xy 회전)
        if self.aug_yaw_random:
            yaw = np.random.uniform(-np.pi, np.pi)
            c, s = np.cos(yaw), np.sin(yaw)
            x = out[..., 0].copy(); y = out[..., 1].copy()
            out[..., 0] = c * x - s * y
            out[..., 1] = s * x + c * y

        # 2. Jitter (xyz 가우시안)
        if self.aug_jitter_std > 0:
            sigma_norm = self.aug_jitter_std / max(self.roi_radius, 1e-6)
            jitter = np.random.normal(0.0, sigma_norm, size=out[..., :3].shape).astype(np.float32)
            out[..., :3] += jitter

        # 3. Point dropout
        if self.aug_dropout_p > 0:
            mask = np.random.rand(*out.shape[:2]) < self.aug_dropout_p
            out[mask] = 0.0

        # 4. Doppler scale (V)
        if self.aug_doppler_scale > 0:
            scale = 1.0 + np.random.uniform(-self.aug_doppler_scale, self.aug_doppler_scale)
            out[..., 3] *= scale

        # 5. v2 신규: Z-scale aug
        if self.aug_z_scale_range is not None:
            zmin, zmax = self.aug_z_scale_range
            z_scale = np.random.uniform(zmin, zmax)
            # padding(0) 은 영향 없음 (0 × scale = 0)
            out[..., 2] *= z_scale

        return out

    def __getitem__(self, idx):
        pts = self.points[idx]
        if self.augment:
            pts = self._augment(pts)
        # metadata 는 augment 적용 후 점들로 계산 (학습-추론 정합).
        meta = extract_metadata_np(pts)
        return (
            torch.from_numpy(pts).float(),
            torch.from_numpy(meta).float(),
            torch.tensor(self.human[idx], dtype=torch.float32),
            torch.tensor(self.pose[idx],  dtype=torch.long),
        )


def make_balanced_sampler(human, pose,
                           w_human=1.0, w_animal=3.0, w_clutter=2.0,
                           w_low_human=4.0):
    """v2: pobok 사람 (human=1 & pose==2) 별도 가중치."""
    weights = np.empty(len(human), dtype=np.float64)
    for i in range(len(human)):
        if human[i] == 1:
            if pose[i] == 2:           # pobok
                weights[i] = w_low_human
            else:
                weights[i] = w_human
        elif pose[i] >= 0:
            weights[i] = w_animal
        else:
            weights[i] = w_clutter

    # 가중치 분포 진단 출력
    n_low = int(((human == 1) & (pose == 2)).sum())
    n_h_other = int(((human == 1) & (pose != 2)).sum())
    n_a = int(((human == 0) & (pose >= 0)).sum())
    n_c = int(((human == 0) & (pose < 0)).sum())
    eff_total = n_low * w_low_human + n_h_other * w_human + n_a * w_animal + n_c * w_clutter
    print(f'[sampler] effective weights:')
    print(f'  low_human (pobok): n={n_low:>5d} × w={w_low_human:.1f} = {n_low*w_low_human:>7.1f} '
          f'({n_low*w_low_human/eff_total*100:.1f}%)')
    print(f'  other  human    : n={n_h_other:>5d} × w={w_human:.1f} = {n_h_other*w_human:>7.1f} '
          f'({n_h_other*w_human/eff_total*100:.1f}%)')
    print(f'  animal (dog)    : n={n_a:>5d} × w={w_animal:.1f} = {n_a*w_animal:>7.1f} '
          f'({n_a*w_animal/eff_total*100:.1f}%)')
    print(f'  clutter (noise) : n={n_c:>5d} × w={w_clutter:.1f} = {n_c*w_clutter:>7.1f} '
          f'({n_c*w_clutter/eff_total*100:.1f}%)')

    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights), replacement=True,
    )


## 7. Multi-block in-session split

v1 과 동일 — 각 recording 을 6 chunk 로 나눠 4/1/1 무작위 분배. gap=20 으로 누수 방지.


In [ ]:
def make_multi_block_split(recording, window_start, n_blocks=6, train_blocks=4,
                           val_blocks=1, gap=20, seed=42):
    train, val, test = [], [], []
    test_blocks = n_blocks - train_blocks - val_blocks
    assert test_blocks >= 1
    half_gap = gap // 2
    for rec in np.unique(recording):
        mask = (recording == rec)
        idx_global = np.where(mask)[0]
        order = np.argsort(window_start[idx_global])
        idx_sorted = idx_global[order]
        n = len(idx_sorted)
        if n < n_blocks * 8:
            train.extend(idx_sorted.tolist())
            continue
        boundaries = [int(n * i / n_blocks) for i in range(n_blocks + 1)]
        rec_rng = np.random.default_rng(hash(str(rec)) % (2**32) ^ seed)
        block_order = np.arange(n_blocks)
        rec_rng.shuffle(block_order)
        for i, bi in enumerate(block_order):
            start = boundaries[bi] + half_gap
            end   = boundaries[bi + 1] - half_gap
            if start >= end:
                continue
            chunk = idx_sorted[start:end].tolist()
            if i < train_blocks:
                train.extend(chunk)
            elif i < train_blocks + val_blocks:
                val.extend(chunk)
            else:
                test.extend(chunk)
    return np.array(sorted(train)), np.array(sorted(val)), np.array(sorted(test))


data = np.load(DATA_PATH, allow_pickle=True)
print(f"NPZ shape  : {data['points'].shape}")
print(f"N total    : {len(data['human'])}")
print(f"  human=1 (사람)   : {int((data['human'] == 1).sum())}")
print(f"  human=0 (비사람) : {int((data['human'] == 0).sum())}")
print(f"  pose=0 (upright) : {int((data['pose'] == 0).sum())}")
print(f"  pose=1 (horiz)   : {int((data['pose'] == 1).sum())}")
print(f"  pose=2 (low)     : {int((data['pose'] == 2).sum())}")
print(f"  pobok (h=1,p=2)  : {int(((data['human']==1) & (data['pose']==2)).sum())}")

if 'window_start' not in data.files:
    raise RuntimeError('window_start 키가 NPZ 에 없습니다. v6 이상 NPZ 필요.')

train_idx, val_idx, test_idx = make_multi_block_split(
    data['recording'], data['window_start'],
    n_blocks=6, train_blocks=4, val_blocks=1, gap=20, seed=SEED,
)
print(f"\nsplits     : train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    n_h = int((data['human'][idx] == 1).sum())
    n_d = int((data['human'][idx] == 0).sum())
    n_pobok = int(((data['human'][idx]==1) & (data['pose'][idx]==2)).sum())
    print(f"  {name:5s}: total={len(idx):>5d}  h=1={n_h:>5d}  h=0={n_d:>4d}  pobok={n_pobok:>4d}")


## 8. DataLoader + v2 Sampler (pobok 가중치)

In [ ]:
train_ds = SequenceDatasetCascade(
    DATA_PATH, indices=train_idx, augment=True,
    aug_jitter_std=AUG_JITTER_STD,
    aug_dropout_p=AUG_DROPOUT_P,
    aug_doppler_scale=AUG_DOPPLER_SCALE,
    aug_yaw_random=AUG_YAW_RANDOM,
    roi_radius=ROI_RADIUS,
    aug_z_scale_range=(AUG_Z_SCALE_MIN, AUG_Z_SCALE_MAX),   # v2 신규
)
val_ds  = SequenceDatasetCascade(DATA_PATH, indices=val_idx, augment=False)
test_ds = SequenceDatasetCascade(DATA_PATH, indices=test_idx, augment=False)

sampler = make_balanced_sampler(
    data['human'][train_idx], data['pose'][train_idx],
    w_human=W_CLASS_HUMAN, w_animal=W_CLASS_ANIMAL,
    w_clutter=W_CLASS_CLUTTER, w_low_human=W_CLASS_LOW_HUMAN,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
val_loader   = DataLoader(val_ds,  batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

print(f'\ntrain_loader: {len(train_loader)} batches × BS={BATCH_SIZE}')
print(f'val_loader  : {len(val_loader)} batches')
print(f'test_loader : {len(test_loader)} batches')

# metadata 8차원 sanity check
print('\nmetadata sample (3 train):')
for i in range(3):
    _, meta, _, _ = train_ds[i]
    assert meta.shape[0] == METADATA_DIM, f'METADATA_DIM mismatch: {meta.shape[0]} vs {METADATA_DIM}'
    print(f'  {meta.numpy().round(3)}  (dim={meta.shape[0]})')


## 9. 학습 / 평가 함수

In [ ]:
def focal_loss_fn(logits, targets, alpha=0.25, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    p_t = torch.exp(-bce)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    focal = alpha_t * (1 - p_t) ** gamma * bce
    return focal.mean()


def train_one_epoch(model, loader, optimizer, device,
                    weight_human=0.7, weight_pose=0.3, grad_clip=1.0,
                    use_focal=False, focal_alpha=0.25, focal_gamma=2.0,
                    pose_label_smoothing=0.0):
    model.train()
    bce = nn.BCEWithLogitsLoss()
    ce  = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=pose_label_smoothing)
    total_loss = total_h_loss = total_p_loss = 0.0
    n = 0
    for pts, meta, human, pose in loader:
        pts  = pts.to(device, non_blocking=True)
        meta = meta.to(device, non_blocking=True)
        human = human.to(device, non_blocking=True)
        pose  = pose.to(device,  non_blocking=True)
        optimizer.zero_grad()
        h_logit, p_logits = model(pts, meta)
        if use_focal:
            loss_h = focal_loss_fn(h_logit, human, focal_alpha, focal_gamma)
        else:
            loss_h = bce(h_logit, human)
        loss_p = ce(p_logits, pose)
        loss = weight_human * loss_h + weight_pose * loss_p
        loss.backward()
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        bs = pts.size(0)
        total_loss   += loss.item()   * bs
        total_h_loss += loss_h.item() * bs
        total_p_loss += loss_p.item() * bs
        n += bs
    return {'loss': total_loss/max(n,1),
            'human_loss': total_h_loss/max(n,1),
            'pose_loss': total_p_loss/max(n,1)}


@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    logits_list, human_list, pose_pred_list, pose_list = [], [], [], []
    for pts, meta, human, pose in loader:
        pts = pts.to(device, non_blocking=True)
        meta = meta.to(device, non_blocking=True)
        h_logit, p_logits = model(pts, meta)
        logits_list.append(h_logit.detach().cpu().numpy())
        human_list.append(human.numpy())
        pose_pred_list.append(p_logits.argmax(dim=1).detach().cpu().numpy())
        pose_list.append(pose.numpy())
    logits = np.concatenate(logits_list)
    human  = np.concatenate(human_list).astype(int)
    pose_pred = np.concatenate(pose_pred_list)
    pose = np.concatenate(pose_list)
    probs = 1.0 / (1.0 + np.exp(-logits))
    pred = (probs > threshold).astype(int)
    tp = int(((pred == 1) & (human == 1)).sum())
    fp = int(((pred == 1) & (human == 0)).sum())
    fn = int(((pred == 0) & (human == 1)).sum())
    tn = int(((pred == 0) & (human == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-6)
    accuracy = (tp + tn) / max(len(human), 1)
    n_h_total  = int((human == 1).sum())
    n_nh_total = int((human == 0).sum())
    h_acc  = tp / max(n_h_total, 1)
    nh_acc = tn / max(n_nh_total, 1)
    valid = pose >= 0
    pose_acc = float((pose_pred[valid] == pose[valid]).mean()) if valid.any() else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': accuracy, 'pose_acc': pose_acc,
            'h_acc': h_acc, 'nh_acc': nh_acc,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}


## 10. 학습 실행

저장 위치: `MyDrive/output_cas_v2/`


In [ ]:
import time, json
from torch.utils.tensorboard import SummaryWriter

model = HumanPoseModelCascade(metadata_dim=METADATA_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
writer = SummaryWriter(log_dir=str(OUT_DIR / 'tb'))

with open(OUT_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'arch': 'cascade_v2',
        'metadata_dim': METADATA_DIM,
        'metadata_features': ['z_mean', 'z_std', 'z_min',
                              'v_abs_mean', 'p_mean', 'p_std',
                              'x_spread', 'y_spread'],
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'weight_human': WEIGHT_HUMAN,
        'weight_pose': WEIGHT_POSE, 'patience': PATIENCE,
        'min_epochs': MIN_EPOCHS, 'early_stop_metric': EARLY_STOP_METRIC,
        'threshold': THRESHOLD, 'seed': SEED,
        'aug_jitter': AUG_JITTER_STD, 'aug_dropout': AUG_DROPOUT_P,
        'aug_doppler': AUG_DOPPLER_SCALE, 'aug_yaw': AUG_YAW_RANDOM,
        'aug_z_scale': [AUG_Z_SCALE_MIN, AUG_Z_SCALE_MAX],
        'sampler_weights': {
            'human': W_CLASS_HUMAN, 'animal': W_CLASS_ANIMAL,
            'clutter': W_CLASS_CLUTTER, 'low_human': W_CLASS_LOW_HUMAN,
        },
        'use_focal_loss': USE_FOCAL_LOSS,
        'focal_alpha': FOCAL_ALPHA, 'focal_gamma': FOCAL_GAMMA,
        'model_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'device': str(DEVICE),
        'n_train': len(train_idx), 'n_val': len(val_idx), 'n_test': len(test_idx),
    }, f, ensure_ascii=False, indent=2)


def _score_for_stopping(vm, metric_name):
    if metric_name == 'f1':       return vm['f1']
    if metric_name == 'nh_acc':   return vm['nh_acc']
    if metric_name == 'combined': return 0.5 * vm['f1'] + 0.5 * vm['nh_acc']
    return vm['f1']


history = []
best_score = 0.0
best_metrics = None
best_epoch = -1
patience_left = PATIENCE
t_start = time.time()

print(f"{'epoch':>5} {'loss':>7} {'P':>6} {'R':>6} {'F1':>6} {'h_acc':>6} {'nh_acc':>6} {'pose':>6} {'lr':>9} {'sec':>5}")

for epoch in range(EPOCHS):
    t0 = time.time()
    tr = train_one_epoch(model, train_loader, optimizer, DEVICE,
                          weight_human=WEIGHT_HUMAN, weight_pose=WEIGHT_POSE,
                          grad_clip=GRAD_CLIP,
                          use_focal=USE_FOCAL_LOSS,
                          focal_alpha=FOCAL_ALPHA, focal_gamma=FOCAL_GAMMA,
                          pose_label_smoothing=POSE_LABEL_SMOOTH)
    scheduler.step()
    vm = evaluate(model, val_loader, DEVICE, threshold=THRESHOLD)
    elapsed = time.time() - t0
    lr_now = optimizer.param_groups[0]['lr']

    rec = {'epoch': epoch, 'lr': lr_now,
           **{f'train_{k}': v for k, v in tr.items()},
           **{f'val_{k}':   v for k, v in vm.items()},
           'elapsed_s': elapsed}
    history.append(rec)

    writer.add_scalar('train/loss',       tr['loss'],       epoch)
    writer.add_scalar('train/human_loss', tr['human_loss'], epoch)
    writer.add_scalar('train/pose_loss',  tr['pose_loss'],  epoch)
    writer.add_scalar('val/precision',    vm['precision'],  epoch)
    writer.add_scalar('val/recall',       vm['recall'],     epoch)
    writer.add_scalar('val/f1',           vm['f1'],         epoch)
    writer.add_scalar('val/pose_acc',     vm['pose_acc'],   epoch)
    writer.add_scalar('val/h_acc',        vm['h_acc'],      epoch)
    writer.add_scalar('val/nh_acc',       vm['nh_acc'],     epoch)
    writer.add_scalar('lr',               lr_now,           epoch)

    print(f"{epoch+1:>5} {tr['loss']:>7.4f} {vm['precision']:>6.3f} "
          f"{vm['recall']:>6.3f} {vm['f1']:>6.3f} "
          f"{vm['h_acc']:>6.3f} {vm['nh_acc']:>6.3f} "
          f"{vm['pose_acc']:>6.3f} "
          f"{lr_now:>9.2e} {elapsed:>5.1f}", flush=True)

    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_metrics': vm,
                'arch': 'cascade_v2', 'metadata_dim': METADATA_DIM,
                }, OUT_DIR / 'last.pt')

    score = _score_for_stopping(vm, EARLY_STOP_METRIC)
    if score > best_score:
        best_score = score
        best_metrics = vm.copy()
        best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_metrics': vm,
                    'arch': 'cascade_v2', 'metadata_dim': METADATA_DIM,
                    }, OUT_DIR / 'best.pt')
        patience_left = PATIENCE
    else:
        if epoch + 1 > MIN_EPOCHS:
            patience_left -= 1
            if patience_left <= 0:
                print(f"\n[early-stop] epoch {epoch+1}  best_{EARLY_STOP_METRIC}={best_score:.3f}  "
                      f"(P={best_metrics['precision']:.3f} R={best_metrics['recall']:.3f} "
                      f"F1={best_metrics['f1']:.3f} nh_acc={best_metrics['nh_acc']:.3f} "
                      f"@ epoch {best_epoch+1})")
                break

    with open(OUT_DIR / 'history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

writer.close()
total_min = (time.time() - t_start) / 60.0
if best_metrics is not None:
    print(f"\n[done] best epoch {best_epoch+1}  {EARLY_STOP_METRIC}={best_score:.4f}")
    print(f"       P={best_metrics['precision']:.3f}  R={best_metrics['recall']:.3f}  F1={best_metrics['f1']:.3f}")
    print(f"       h_acc={best_metrics['h_acc']:.3f}  nh_acc={best_metrics['nh_acc']:.3f}")
print(f"[done] total time = {total_min:.1f} min")
print(f"[done] checkpoints in {OUT_DIR}")


## 11. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUT_DIR/tb


## 12. Test 평가 — 전체 (best.pt 로드)


In [ ]:
POSE_NAMES = ['upright', 'horizontal', 'low']

ckpt = torch.load(OUT_DIR / 'best.pt', map_location=DEVICE)
model = HumanPoseModelCascade(metadata_dim=METADATA_DIM).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"loaded best.pt (epoch {ckpt['epoch']+1}, val_P={ckpt['val_metrics']['precision']:.3f})")

# test 전체 inference (인덱스 보존)
all_logits, all_human, all_pose_pred, all_pose = [], [], [], []
with torch.no_grad():
    for pts, meta, human, pose in test_loader:
        pts = pts.to(DEVICE); meta = meta.to(DEVICE)
        h_logit, p_logits = model(pts, meta)
        all_logits.append(h_logit.detach().cpu().numpy())
        all_human.append(human.numpy())
        all_pose_pred.append(p_logits.argmax(dim=1).detach().cpu().numpy())
        all_pose.append(pose.numpy())
logits    = np.concatenate(all_logits)
human     = np.concatenate(all_human).astype(int)
pose_pred = np.concatenate(all_pose_pred)
pose      = np.concatenate(all_pose)
probs     = 1.0 / (1.0 + np.exp(-logits))

# test set 의 recording 정보 (세션별 평가용)
test_recording = data['recording'][test_idx]
test_track_id  = data['track_id'][test_idx] if 'track_id' in data.files else np.zeros(len(test_idx), dtype=int)
DOG_PREFIX = 100000

# ROC AUC
def roc_auc(probs, human):
    order = np.argsort(-probs)
    y_sorted = human[order]
    n_pos = max(int(human.sum()), 1)
    n_neg = max(int(len(human) - human.sum()), 1)
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    tpr = np.concatenate([[0], tp / n_pos])
    fpr = np.concatenate([[0], fp / n_neg])
    return float(np.trapz(tpr, fpr)), tpr, fpr

auc, tpr, fpr = roc_auc(probs, human)
print(f'\nROC AUC = {auc:.4f}')

n_human    = int((human == 1).sum())
n_nonhuman = int((human == 0).sum())
print(f'test set : human={n_human}  non-human={n_nonhuman}')

# threshold sweep
print(f"\n{'thresh':>7} {'P':>6} {'R':>6} {'F1':>6} {'h_acc':>6} {'nh_acc':>6} {'TP':>5} {'FP':>5} {'FN':>5} {'TN':>5}")
target_row = None
sweep_results = []
for t in np.linspace(0.05, 0.95, 19):
    pred = (probs > t).astype(int)
    tp = int(((pred == 1) & (human == 1)).sum())
    fp = int(((pred == 1) & (human == 0)).sum())
    fn = int(((pred == 0) & (human == 1)).sum())
    tn = int(((pred == 0) & (human == 0)).sum())
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    f1 = 2 * p * r / max(p + r, 1e-6)
    h_acc  = tp / max(n_human, 1)
    nh_acc = tn / max(n_nonhuman, 1)
    sweep_results.append({'threshold': float(t), 'precision': p, 'recall': r,
                          'f1': f1, 'h_acc': h_acc, 'nh_acc': nh_acc,
                          'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})
    mark = ''
    if target_row is None and p >= 0.95:
        target_row = sweep_results[-1]
        mark = '  ← P≥0.95 첫 지점'
    print(f"{t:>7.2f} {p:>6.3f} {r:>6.3f} {f1:>6.3f} "
          f"{h_acc:>6.3f} {nh_acc:>6.3f} "
          f"{tp:>5d} {fp:>5d} {fn:>5d} {tn:>5d}{mark}")

# 추천 threshold
print()
if target_row:
    rrow = target_row
    print(f"=== threshold={rrow['threshold']:.2f} (P≥0.95 첫 지점) ===")
else:
    rrow = max(sweep_results, key=lambda x: x['f1'])
    print(f"=== threshold={rrow['threshold']:.2f} (best F1) ===")
print(f"  사람        : {rrow['tp']:>5d} / {n_human:>5d} = {rrow['h_acc']:.3f}  (recall)")
print(f"  비사람       : {rrow['tn']:>5d} / {n_nonhuman:>5d} = {rrow['nh_acc']:.3f}  (specificity)")


## 12-bis. 합격 조건 평가 — Pobok recall + Dog FPR

handoff §7 의 핵심 합격 조건:
- **B. Pobok recall ≥ 0.70** (h=1, p=2 sample 에서 sigmoid > 0.5 비율)
- **C. Dog FPR < 0.10** (dog ns sample 에서 사람으로 분류 비율)


In [ ]:
ALARM_THRESHOLD = 0.5    # handoff 권장

# === Pobok recall (h=1, pose=2) ===
pobok_mask = (human == 1) & (pose == 2)
n_pobok = int(pobok_mask.sum())
if n_pobok > 0:
    pobok_probs = probs[pobok_mask]
    pobok_recall = float((pobok_probs > ALARM_THRESHOLD).mean())
    print(f'=== Pobok 합격 조건 ===')
    print(f'  n_pobok = {n_pobok}')
    print(f'  sigmoid > {ALARM_THRESHOLD} 비율 = {pobok_recall:.3f}')
    print(f'  합격 기준 = 0.70')
    print(f'  → {"✓ PASS" if pobok_recall >= 0.70 else "✗ FAIL"}')
    print(f'  pobok prob 분포: min={pobok_probs.min():.3f} '
          f'p25={np.percentile(pobok_probs,25):.3f} '
          f'p50={np.percentile(pobok_probs,50):.3f} '
          f'p75={np.percentile(pobok_probs,75):.3f} '
          f'max={pobok_probs.max():.3f}')
else:
    pobok_recall = None
    print('[skip] test set 에 pobok 없음 (h=1 ∩ p=2)')

# === Dog FPR (track_id >= DOG_PREFIX) ===
print(f'\n=== Dog FPR 합격 조건 ===')
dog_mask = test_track_id >= DOG_PREFIX
n_dog = int(dog_mask.sum())
if n_dog > 0:
    dog_probs = probs[dog_mask]
    dog_fpr = float((dog_probs > ALARM_THRESHOLD).mean())
    print(f'  n_dog (track_id >= {DOG_PREFIX}) = {n_dog}')
    print(f'  sigmoid > {ALARM_THRESHOLD} 비율 = {dog_fpr:.3f}')
    print(f'  합격 기준 < 0.10')
    print(f'  → {"✓ PASS" if dog_fpr < 0.10 else "✗ FAIL"}')
else:
    dog_fpr = None
    # fallback: human=0 & pose>=0 추정 dog
    dog_mask = (human == 0) & (pose >= 0)
    n_dog = int(dog_mask.sum())
    if n_dog > 0:
        dog_probs = probs[dog_mask]
        dog_fpr = float((dog_probs > ALARM_THRESHOLD).mean())
        print(f'  [fallback: h=0 & p>=0 으로 dog 추정] n={n_dog}')
        print(f'  sigmoid > {ALARM_THRESHOLD} 비율 = {dog_fpr:.3f}  '
              f'→ {"✓ PASS" if dog_fpr < 0.10 else "✗ FAIL"}')


## 12-tri. 세션별 성능 평가

각 recording 별로 사람/비사람 recall, pobok recall (해당되는 경우) 출력.
회귀 감지용 — 특정 세션에서만 성능이 깨지면 알 수 있음.


In [ ]:
# recording 을 카테고리 수준으로 그룹핑 (앞 1~2 path component)
def session_key(rec):
    parts = rec.split('/')
    # 'pobok/pobok9/...' → 'pobok/pobok9'
    # '515_dog_and_human/dah1/...' → '515_dog_and_human/dah1'
    return '/'.join(parts[:2]) if len(parts) >= 2 else parts[0]

session_keys = np.array([session_key(r) for r in test_recording])
unique_sessions = sorted(set(session_keys.tolist()))

session_results = []
print(f"{'session':45s} {'n':>5s} {'h':>5s} {'nh':>5s} "
      f"{'h_acc':>7s} {'nh_acc':>7s} {'pobok_R':>8s} {'pose_acc':>9s}")
print('-' * 100)
for sess in unique_sessions:
    m = session_keys == sess
    n = int(m.sum())
    if n == 0: continue
    sess_human = human[m]
    sess_pose = pose[m]
    sess_probs = probs[m]
    sess_pose_pred = pose_pred[m]

    nh = int((sess_human == 1).sum())
    nnh = int((sess_human == 0).sum())
    pred = (sess_probs > ALARM_THRESHOLD).astype(int)

    if nh > 0:
        h_acc = float(((pred == 1) & (sess_human == 1)).sum() / nh)
    else:
        h_acc = None
    if nnh > 0:
        nh_acc = float(((pred == 0) & (sess_human == 0)).sum() / nnh)
    else:
        nh_acc = None

    # 세션 내 pobok recall (해당하면)
    pobok_m_sess = (sess_human == 1) & (sess_pose == 2)
    n_pob = int(pobok_m_sess.sum())
    if n_pob > 0:
        pobok_r = float((sess_probs[pobok_m_sess] > ALARM_THRESHOLD).mean())
    else:
        pobok_r = None

    valid = sess_pose >= 0
    if valid.any():
        pose_acc_sess = float((sess_pose_pred[valid] == sess_pose[valid]).mean())
    else:
        pose_acc_sess = None

    session_results.append({
        'session': sess, 'n': n, 'n_human': nh, 'n_nonhuman': nnh,
        'h_acc': h_acc, 'nh_acc': nh_acc,
        'pobok_recall': pobok_r, 'n_pobok': n_pob,
        'pose_acc': pose_acc_sess,
    })
    print(f"{sess:45s} {n:>5d} {nh:>5d} {nnh:>5d} "
          f"{'-' if h_acc is None else f'{h_acc:>7.3f}'} "
          f"{'-' if nh_acc is None else f'{nh_acc:>7.3f}'} "
          f"{'-' if pobok_r is None else f'{pobok_r:>6.3f}({n_pob})'} "
          f"{'-' if pose_acc_sess is None else f'{pose_acc_sess:>9.3f}'}")


## 12-quad. Class × Pose 교차 분석

각 (human, pose) 조합별 모델 출력 분포 — 어떤 cell 이 약한지 한눈에 확인.


In [ ]:
# (human, pose) 조합별 평균 sigmoid + 정확도
print(f"{'human':>6s} {'pose':>12s} {'n':>5s} {'mean_prob':>10s} {'std_prob':>9s} "
      f"{'pred=1 비율':>11s} {'pose_acc':>9s}")
print('-' * 80)
class_pose_stats = []
for h_val in [1, 0]:
    for p_val in [0, 1, 2, -1]:
        m = (human == h_val) & (pose == p_val)
        n = int(m.sum())
        if n == 0:
            continue
        pp = probs[m]
        pred = (pp > ALARM_THRESHOLD).astype(int)
        pred_rate = float(pred.mean())
        ppose = pose_pred[m]
        if p_val >= 0:
            pose_acc_cell = float((ppose == p_val).mean())
        else:
            pose_acc_cell = None
        pose_name = POSE_NAMES[p_val] if p_val >= 0 else 'unknown'
        h_label = '사람' if h_val == 1 else '비사람'
        class_pose_stats.append({
            'human': h_val, 'pose': p_val, 'pose_name': pose_name,
            'n': n, 'mean_prob': float(pp.mean()), 'std_prob': float(pp.std()),
            'pred_human_rate': pred_rate, 'pose_acc': pose_acc_cell,
        })
        print(f"{h_label:>6s} {pose_name:>12s} {n:>5d} "
              f"{pp.mean():>10.3f} {pp.std():>9.3f} {pred_rate:>11.3f} "
              f"{'-' if pose_acc_cell is None else f'{pose_acc_cell:>9.3f}'}")

# 핵심 셀 강조
print('\n=== 핵심 셀 ===')
for r in class_pose_stats:
    if r['human'] == 1 and r['pose'] == 2:
        print(f'  [pobok]    h=1, pose=low : n={r["n"]} '
              f'mean_prob={r["mean_prob"]:.3f}  '
              f'pred=human 비율={r["pred_human_rate"]:.3f}  '
              f'→ {"✓" if r["pred_human_rate"]>=0.70 else "✗"}')
    if r['human'] == 0 and r['pose'] >= 0:
        print(f'  [dog/{r["pose_name"]}] h=0, pose={r["pose_name"]} : n={r["n"]} '
              f'mean_prob={r["mean_prob"]:.3f}  '
              f'pred=human 비율={r["pred_human_rate"]:.3f}  '
              f'→ {"✓" if r["pred_human_rate"]<0.10 else "✗"}')


## 12-pose. Pose confusion matrix


In [ ]:
valid = pose >= 0
cm = np.zeros((3, 3), dtype=int)
for tt, pp in zip(pose[valid], pose_pred[valid]):
    if 0 <= tt < 3 and 0 <= pp < 3:
        cm[tt, pp] += 1
pose_acc = float((pose_pred[valid] == pose[valid]).mean()) if valid.any() else 0.0

print(f"=== pose confusion (valid={int(valid.sum())}/{len(pose)}, acc={pose_acc:.3f}) ===")
print(f"  {'gt\\pred':>12}" + ''.join(f'{n:>12}' for n in POSE_NAMES))
for i, name in enumerate(POSE_NAMES):
    rs = int(cm[i].sum())
    ra = cm[i, i] / max(rs, 1)
    print(f"  {name:>12}" + ''.join(f'{int(cm[i, j]):>12d}' for j in range(3))
          + f'   n={rs} acc={ra:.3f}')

# Eval report 저장
import json as _json
eval_report = {
    'arch': 'cascade_v2',
    'metadata_dim': METADATA_DIM,
    'n_test': len(test_idx),
    'threshold_used': ALARM_THRESHOLD,
    'roc_auc': auc,
    'target_row': target_row,
    'threshold_sweep': sweep_results,
    'pose_confusion': cm.tolist(),
    'pose_accuracy': pose_acc,
    'pose_class_names': POSE_NAMES,
    'pobok_recall': pobok_recall,
    'pobok_pass': bool(pobok_recall is not None and pobok_recall >= 0.70),
    'dog_fpr': dog_fpr,
    'dog_pass': bool(dog_fpr is not None and dog_fpr < 0.10),
    'session_results': session_results,
    'class_pose_breakdown': class_pose_stats,
}
with open(OUT_DIR / 'eval_report.json', 'w', encoding='utf-8') as f:
    _json.dump(eval_report, f, ensure_ascii=False, indent=2)
print(f'\n[saved] {OUT_DIR / "eval_report.json"}')
print(f'[summary] pobok_pass={eval_report["pobok_pass"]}  dog_pass={eval_report["dog_pass"]}')


## 13. ROC 그래프

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC (human) — Cascade v2')
plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / 'roc.png', dpi=120)
plt.show()
print(f"[saved] {OUT_DIR / 'roc.png'}")


## 14. 모델 추출 — TFLite Int8 (TF 2.10 호환) + C 헤더

handoff §6 요건: **반드시 tensorflow==2.10.x 환경에서 변환**. ARM SDK
(libtensorflowlite_c.so) 가 TF 2.10.0 빌드라 신형 schema 와 비호환.

산출물 ([output_cas_v2/embed/]):
- `model_fp32.tflite` (디버깅용)
- `model_int8.tflite` (배포용, Int8 PTQ)
- `tfmodel.h` (C array header)
- `embed_report.json` (parity 검증)


In [ ]:
# Colab 기본 TF 가 2.10 이 아니면 강제로 다운그레이드.
# (이미 학습 끝났으니 변환에만 영향 — TF 만 바꾸고 PyTorch 는 그대로.)
import subprocess, sys
try:
    import tensorflow as tf
    tf_ver = tf.__version__
except ImportError:
    tf_ver = None
print(f'[tf] current: {tf_ver}')
if tf_ver is None or not tf_ver.startswith('2.10'):
    print('[tf] installing 2.10.1 (handoff requires TF 2.10.x for ARM SDK compat)...')
    !pip install -q tensorflow==2.10.1
    print('[tf] ⚠ 런타임 재시작 필요할 수 있음. 재시작 후 셀 14 부터 다시 실행.')
    print('[tf] 재시작 안 해도 import 새로 시도:')
    import importlib
    if 'tensorflow' in sys.modules:
        del sys.modules['tensorflow']
    import tensorflow as tf
    print(f'[tf] now: {tf.__version__}')


In [ ]:
!pip install -q onnx onnx2tf onnx-simplifier


In [ ]:
# best.pt 로드
import json, shutil, time
import tensorflow as tf

EMBED_DIR = OUT_DIR / 'embed'
EMBED_DIR.mkdir(parents=True, exist_ok=True)

best_ckpt = torch.load(str(OUT_DIR / 'best.pt'), map_location=DEVICE, weights_only=False)
if isinstance(best_ckpt, dict) and 'model_state_dict' in best_ckpt:
    model.load_state_dict(best_ckpt['model_state_dict'])
elif isinstance(best_ckpt, dict) and 'model' in best_ckpt:
    model.load_state_dict(best_ckpt['model'])
else:
    model.load_state_dict(best_ckpt)
model.eval()
model_cpu = model.to('cpu')
print(f'[extract] model from best.pt loaded ({sum(p.numel() for p in model_cpu.parameters()):,} params)')
print(f'[extract] METADATA_DIM = {METADATA_DIM} (v2)')


In [ ]:
# Calibration data: train set 500 sample
N_CALIB = 500
rng = np.random.default_rng(42)
calib_idx = rng.choice(train_idx, size=min(N_CALIB, len(train_idx)), replace=False)

pts_calib = data['points'][calib_idx].astype(np.float32)
# v2: 8 차원 metadata
meta_calib = np.stack([extract_metadata_np(p) for p in pts_calib]).astype(np.float32)
assert meta_calib.shape[1] == METADATA_DIM, f'meta dim mismatch: {meta_calib.shape[1]} vs {METADATA_DIM}'
print(f'[calib] pts {pts_calib.shape}, meta {meta_calib.shape}')

sample_pts = torch.from_numpy(pts_calib[:1])
sample_meta = torch.from_numpy(meta_calib[:1])


In [ ]:
# 변환: ONNX → onnx2tf → TFLite (handoff 권장 경로)
fp32_path = EMBED_DIR / 'model_fp32.tflite'
int8_path = EMBED_DIR / 'model_int8.tflite'
onnx_path = EMBED_DIR / 'model.onnx'

# dynamo=False: PyTorch 2.x 신 익스포터는 GRU 가중치를 상수가 아닌 입력으로
#   추출 → onnx2tf flatbuffer_direct path 가 "GRU W/R must be constant" 로 실패.
#   레거시 TorchScript 익스포터는 가중치를 ONNX 상수로 인라인 → 정상.
# 일부 PyTorch 버전은 dynamo 인자가 없으니 fallback 로 한 번 더 시도.
try:
    torch.onnx.export(
        model_cpu, (sample_pts, sample_meta), str(onnx_path),
        input_names=['points', 'metadata'],
        output_names=['human_logit', 'pose_logits'],
        opset_version=18, do_constant_folding=True,
        dynamo=False,
    )
except TypeError:
    # 옛 PyTorch: dynamo 인자 없음 (기본이 이미 레거시)
    torch.onnx.export(
        model_cpu, (sample_pts, sample_meta), str(onnx_path),
        input_names=['points', 'metadata'],
        output_names=['human_logit', 'pose_logits'],
        opset_version=18, do_constant_folding=True,
    )
print(f'[onnx] saved: {onnx_path.stat().st_size/1024:.1f} KB')

# 안전망 — onnx-simplifier 로 GRU weight 상수 폴딩 다시 확인.
# 일부 환경(dynamo 인자 무시 등)에서도 onnxsim 이 fold 해줌.
try:
    import onnxsim, onnx
    simp_model, check = onnxsim.simplify(onnx.load(str(onnx_path)))
    if check:
        onnx.save(simp_model, str(onnx_path))
        print(f'[onnxsim] simplified: {onnx_path.stat().st_size/1024:.1f} KB')
    else:
        print('[onnxsim] check failed — 원본 ONNX 유지')
except ImportError:
    print('[onnxsim] not installed — 건너뜀 (필요시 pip install onnx-simplifier)')

import onnx2tf
tf_dir = EMBED_DIR / 'model_tf'
if tf_dir.exists():
    shutil.rmtree(tf_dir)

# ===== FP32 변환 =====
# 새 onnx2tf: flatbuffer_direct 백엔드 — SavedModel 없이 .tflite 직접 생성
# 옛 onnx2tf: SavedModel 경유 — TFLiteConverter.from_saved_model 필요
# 양쪽 모두 지원.
onnx2tf.convert(
    input_onnx_file_path=str(onnx_path),
    output_folder_path=str(tf_dir),
    not_use_onnxsim=True,
    copy_onnx_input_output_names_to_tflite=True,
    output_signaturedefs=True,
    keep_shape_absolutely_input_names=['points', 'metadata'],
)

direct_fp32 = tf_dir / 'model_float32.tflite'
if direct_fp32.exists():
    shutil.copy(direct_fp32, fp32_path)
    print(f'[fp32] from onnx2tf direct: {fp32_path.stat().st_size/1024:.1f} KB')
else:
    # 옛 onnx2tf — SavedModel 경유
    conv = tf.lite.TFLiteConverter.from_saved_model(str(tf_dir))
    fp32_path.write_bytes(conv.convert())
    print(f'[fp32] from SavedModel: {fp32_path.stat().st_size/1024:.1f} KB')
fp32_kb = fp32_path.stat().st_size / 1024

# ===== Int8 PTQ =====
# 새 경로: onnx2tf 내장 PTQ — calib data 를 .npy 로 저장하고 다시 변환
# 옛 경로: tf.lite.TFLiteConverter + representative_dataset
calib_pts_npy = EMBED_DIR / 'calib_pts.npy'
calib_meta_npy = EMBED_DIR / 'calib_meta.npy'
np.save(calib_pts_npy, pts_calib)
np.save(calib_meta_npy, meta_calib)

int8_tf_dir = EMBED_DIR / 'model_tf_int8'
if int8_tf_dir.exists():
    shutil.rmtree(int8_tf_dir)

int8_done = False
try:
    # 새 onnx2tf — 내장 Int8 PTQ
    onnx2tf.convert(
        input_onnx_file_path=str(onnx_path),
        output_folder_path=str(int8_tf_dir),
        not_use_onnxsim=True,
        copy_onnx_input_output_names_to_tflite=True,
        keep_shape_absolutely_input_names=['points', 'metadata'],
        output_integer_quantized_tflite=True,
        quant_type='per-tensor',   # ARM SDK 호환성 — per-channel 도 시도 가능
        custom_input_op_name_np_data_path=[
            ['points', str(calib_pts_npy), [0.0], [1.0]],
            ['metadata', str(calib_meta_npy), [0.0], [1.0]],
        ],
    )
    # onnx2tf 가 생성한 Int8 tflite 후보 검색
    int8_cands = (
        list(int8_tf_dir.glob('*full_integer*.tflite')) +
        list(int8_tf_dir.glob('*integer_quant*.tflite')) +
        list(int8_tf_dir.glob('*int8*.tflite'))
    )
    if int8_cands:
        shutil.copy(int8_cands[0], int8_path)
        print(f'[int8] from onnx2tf PTQ ({int8_cands[0].name}): '
              f'{int8_path.stat().st_size/1024:.1f} KB')
        int8_done = True
    else:
        print(f'[int8] onnx2tf 가 Int8 파일 안 만듦 — fallback 시도')
        print(f'  int8_tf_dir 내용: {[p.name for p in int8_tf_dir.iterdir()]}')
except Exception as e:
    print(f'[int8] onnx2tf PTQ 실패: {e}')

if not int8_done:
    # 옛 경로 — SavedModel 이 있으면 TF Lite Converter 사용
    sm_pb = tf_dir / 'saved_model.pb'
    if sm_pb.exists():
        def rep_data():
            for i in range(len(pts_calib)):
                yield [pts_calib[i:i+1], meta_calib[i:i+1]]
        conv = tf.lite.TFLiteConverter.from_saved_model(str(tf_dir))
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = rep_data
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
        int8_path.write_bytes(conv.convert())
        print(f'[int8] from SavedModel: {int8_path.stat().st_size/1024:.1f} KB')
    else:
        raise RuntimeError(
            'Int8 변환 실패: onnx2tf 내장 PTQ 도, SavedModel TFLiteConverter 도 사용 불가.\n'
            f'직접 확인: ls {int8_tf_dir} / {tf_dir}'
        )
int8_kb = int8_path.stat().st_size / 1024


In [ ]:
# Parity 검증 — PyTorch FP32 vs TFLite Int8
interp = tf.lite.Interpreter(model_path=str(int8_path))
interp.allocate_tensors()
in_dets, out_dets = interp.get_input_details(), interp.get_output_details()
print('[parity] inputs :', [(d['name'], d['shape'].tolist(), d['dtype']) for d in in_dets])
print('[parity] outputs:', [(d['name'], d['shape'].tolist(), d['dtype']) for d in out_dets])

# Shape 검증 (handoff §6 의 기대값)
for d in in_dets:
    if 'point' in d['name'].lower():
        assert list(d['shape']) == [1, 40, 64, 5], f'points shape {d["shape"]} ≠ (1,40,64,5)'
    elif 'meta' in d['name'].lower():
        assert list(d['shape']) == [1, METADATA_DIM], f'meta shape {d["shape"]} ≠ (1,{METADATA_DIM})'
print(f'[parity] ✓ shape OK: points=(1,40,64,5), metadata=(1,{METADATA_DIM})')

N_PARITY = 20
diffs_h, diffs_p = [], []
for i in range(N_PARITY):
    with torch.no_grad():
        h_pt, p_pt = model_cpu(torch.from_numpy(pts_calib[i:i+1]),
                                torch.from_numpy(meta_calib[i:i+1]))
    h_pt = h_pt.numpy().flatten(); p_pt = p_pt.numpy().flatten()

    for d in in_dets:
        scale, zero = d['quantization']
        arr = meta_calib[i:i+1] if 'meta' in d['name'].lower() else pts_calib[i:i+1]
        if d['dtype'] == np.int8 and scale > 0:
            arr = np.round(arr/scale + zero).clip(-128, 127).astype(np.int8)
        interp.set_tensor(d['index'], arr.astype(d['dtype']))
    interp.invoke()

    h_tf = p_tf = None
    for d in out_dets:
        arr = interp.get_tensor(d['index']).astype(np.float32)
        scale, zero = d['quantization']
        if d['dtype'] == np.int8 and scale > 0:
            arr = (arr - zero) * scale
        if arr.size == 1 or arr.shape[-1] == 1:
            h_tf = arr.flatten()
        else:
            p_tf = arr.flatten()
    if h_tf is not None: diffs_h.append(float(np.abs(h_pt - h_tf).max()))
    if p_tf is not None: diffs_p.append(float(np.abs(p_pt - p_tf).max()))

max_h, max_p = max(diffs_h), max(diffs_p)
mean_h, mean_p = float(np.mean(diffs_h)), float(np.mean(diffs_p))
parity_ok = max_h < 0.5
print(f'[parity] max_diff_human={max_h:.4f} (threshold 0.5) → {"PASS" if parity_ok else "FAIL"}')
print(f'[parity] max_diff_pose ={max_p:.4f}')


In [ ]:
# xxd C 헤더 생성 (model_tflite 심볼 — radar_embed/main.cc 가 가정)
header_path = EMBED_DIR / 'tfmodel.h'
src_tflite = int8_path if int8_path.exists() else fp32_path
raw = src_tflite.read_bytes()
n = len(raw)
lines = [
    f'// Auto-generated from {src_tflite.name} ({n:,} bytes)',
    f'// Cascade v2 (METADATA_DIM={METADATA_DIM})',
    '// extern const unsigned char model_tflite[];',
    '// extern const unsigned int model_tflite_len;',
    '#pragma once',
    '',
    'const unsigned char model_tflite[] __attribute__((aligned(16))) = {',
]
for i in range(0, n, 12):
    lines.append('  ' + ', '.join(f'0x{b:02x}' for b in raw[i:i+12]) + ',')
lines.append('};')
lines.append(f'const unsigned int model_tflite_len = {n};')
header_path.write_text('\n'.join(lines), encoding='utf-8')
header_kb = header_path.stat().st_size / 1024
print(f'[header] {header_path.name}: {header_kb:.1f} KB (source: {src_tflite.name})')


In [ ]:
# Embed report 저장
embed_report = {
    'arch': 'cascade_v2',
    'metadata_dim': METADATA_DIM,
    'metadata_features': ['z_mean', 'z_std', 'z_min',
                          'v_abs_mean', 'p_mean', 'p_std',
                          'x_spread', 'y_spread'],
    'params': sum(p.numel() for p in model_cpu.parameters()),
    'fp32_size_kb': round(fp32_kb, 1),
    'int8_size_kb': round(int8_kb, 1),
    'header_size_kb': round(header_kb, 1),
    'tf_version': tf.__version__,
    'parity': {
        'n_samples': N_PARITY,
        'max_diff_human': max_h,
        'mean_diff_human': mean_h,
        'max_diff_pose': max_p,
        'mean_diff_pose': mean_p,
        'pass': bool(parity_ok),
    },
    # 합격 조건 (handoff §7)
    'pass_checks': {
        'pobok_recall': pobok_recall,
        'pobok_pass': bool(pobok_recall is not None and pobok_recall >= 0.70),
        'dog_fpr': dog_fpr,
        'dog_pass': bool(dog_fpr is not None and dog_fpr < 0.10),
        'roc_auc': auc,
        'parity_pass': bool(parity_ok),
    },
}
with open(EMBED_DIR / 'embed_report.json', 'w', encoding='utf-8') as f:
    json.dump(embed_report, f, ensure_ascii=False, indent=2)

print('\n=== 추출 완료 ===')
print(f'  산출물:            {EMBED_DIR}')
print(f'  model_int8.tflite: {int8_kb:.1f} KB')
print(f'  tfmodel.h:         {header_kb:.1f} KB')
print(f'  parity max_diff_h: {max_h:.4f}  → {"PASS" if parity_ok else "FAIL"}')
print(f'  pobok_recall:      '
      f'{("✓" if embed_report["pass_checks"]["pobok_pass"] else "✗") + " " + (f"{pobok_recall:.3f}" if pobok_recall else "-")}')
print(f'  dog_fpr:           '
      f'{("✓" if embed_report["pass_checks"]["dog_pass"] else "✗") + " " + (f"{dog_fpr:.3f}" if dog_fpr else "-")}')
print(f'\n다음 단계 (handoff §9):')
print(f'  1. radar_embed/tfmodel.h 새 파일로 교체')
print(f'  2. radar_embed/include/radar/config.h:71 → METADATA_DIM = {METADATA_DIM}')
print(f'  3. radar_embed/src/metadata.cc::extract() → 8 차원만 채우게 수정')
print(f'     순서: [z_mean, z_std, z_min, v_abs_mean, p_mean, p_std, x_spread, y_spread]')
print(f'  4. main.cc 후처리 휴리스틱 (포복 구제 if) 제거 검토')
print(f'  5. make TARGET=arm → 라이브 테스트')


## 15. 다운로드 (선택)


In [ ]:
try:
    from google.colab import files
    files.download(str(EMBED_DIR / 'tfmodel.h'))
    files.download(str(EMBED_DIR / 'model_int8.tflite'))
    files.download(str(EMBED_DIR / 'embed_report.json'))
    files.download(str(OUT_DIR / 'eval_report.json'))
except ImportError:
    print('local: 산출물은 Drive 의 output_cas_v2/ 에 있음')


## 끝 — v2 산출물 정리

### Drive 산출물
```
MyDrive/output_cas_v2/
├── best.pt                — PyTorch 가중치 (best epoch)
├── config.json            — 학습 설정 + v2 변경사항
├── eval_report.json       — 전체 + 세션별 + class별 + pobok/dog 합격
├── history.json           — epoch 별 학습 로그
├── roc.png                — ROC 곡선
├── tb/                    — TensorBoard 로그
└── embed/
    ├── model_fp32.tflite
    ├── model_int8.tflite
    ├── tfmodel.h          — C array header
    └── embed_report.json  — parity + 합격 결과
```

### v1 vs v2 비교

| 지표 | v1 (output_cas) | v2 (output_cas_v2) |
|---|---|---|
| METADATA_DIM | 11 | 8 |
| Pobok recall | ~0.30 | 목표 ≥ 0.70 |
| Dog FPR | ~0.05 | 목표 < 0.10 |
| ROC AUC | 0.995 | 회귀 없음 (목표) |

### C++ 동기화 체크리스트

배포 전에 반드시:
- [ ] `radar_embed/include/radar/config.h:71` → `METADATA_DIM = 8`
- [ ] `radar_embed/src/metadata.cc::extract()` → `out[0..7]` 만 채움
      순서: `[z_mean, z_std, z_min, v_abs_mean, p_mean, p_std, x_spread, y_spread]`
- [ ] `tfmodel.h` 새 파일로 교체
- [ ] `radar_embed/main.cc:184-188` 포복 구제 if 제거 검토 (모델이 잘 분류하면 불필요)
- [ ] `make TARGET=arm` → 라이브 테스트

### 라이브 검증 시나리오 (handoff §7.D)

1. 포복 사람 → 라벨 `human 70%+ (seq)` 출력 → ✓
2. 서서 걷는 사람 → 라벨 `human 90%+` 유지 (regression 없음) → ✓
3. 강아지 → 사람으로 안 잡힘 → ✓
